# Trading-Crab end to end: build → features → weights → page

One pass through the whole skeleton on the data in this checkout's `data/`.

**Before you run it:**

1. `pip install -e src/trading_crab_lib -e ".[notebook]"` in the venv you start `jupyter lab` from.
2. `.env` holds `FRED_API_KEY` and `TIINGO_API_KEY` (copied from `.env.example`, never committed).
3. `tradingcrab build` has run in this checkout.

**Naive skeleton: nothing here is evidence.** Every layer is deliberately simple (MODULE-MAP.md); the cells only call
the library and print.

In [ ]:
from datetime import date

import pandas as pd

from trading_crab_lib.platform import checkpoints, report
from trading_crab_lib.platform.code_version import code_version
from trading_crab_lib.platform.config import load

print(code_version())  # which checkout's code this kernel runs (G-16)
cfg = load()  # holds the API keys: show it only through config.redacted

## The build and the holdout fence

`report.inputs` reads every built month (the page is live scoring); `checkpoints.load` reads only the development rows,
on or before the 2021+ holdout cutoff. Lags were applied once, at the build: the `unrate` value at a month-end is the
previous month's release (lag 1 in `config/platform_settings.yaml`).

In [ ]:
raw, feats, weights = report.inputs(cfg)
dev = checkpoints.load("monthly_raw")  # development rows only
cutoff = pd.Timestamp(cfg["holdout"]["cutoff"])
assert dev.index.max() <= cutoff, "development rows run past the holdout cutoff"
print("Last month-end with a value, per column:")
print(raw.apply(pd.Series.last_valid_index))
print(f"Development rows: {len(dev)} (through {dev.index.max():%Y-%m-%d}); holdout rows: {(raw.index > cutoff).sum()}")

In [ ]:
# The causal backdrop: market context, not strategy performance. These features set no weight.
print(feats.tail(6).round(4))

In [ ]:
# The target mix: inverse-vol on the last 36 month-end returns, scaled to the vol target, the rest in cash.
print(weights.round(4))
print(f"Sum: {weights.sum():.12f}")

In [ ]:
print(report.render(raw, feats, weights, cfg=cfg, code_line=code_version(), run_date=date.today()))

## Sign-off

Read this notebook beside `outputs/reports/weekly_report.md`, then fill in:

- **Date:**
- **Verdict:**
- **Reasoning:**